# V0.7 Process Runtime Lab

问题：一个 Process 预算耗尽，为什么 Agent authority 仍然独立？

这个 notebook 逐格展示 Agent、Process、Session 的边界，以及 scheduler safe point 的预算阻塞。

## Mode

`deterministic`：Model decision = SCRIPTED，Kernel execution = REAL。

`real_model`：Model decision = REAL OpenAI-compatible，Kernel execution = REAL。优先使用 `AGENTKERNEL_LAB_LLM_*`，否则复用仓库本地 `.minicode/config.json`。不会展示 hidden chain-of-thought，也不会展示 API key。

In [ ]:
MODE = "deterministic"

from pathlib import Path
import sys

def find_agentkernel_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir() and (path / "labs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the AgentKernel repo root or the labs directory.")

REPO_ROOT = find_agentkernel_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from labs import create_lab

lab = create_lab("v07", mode=MODE)


## Step 1: Setup

创建 Agent、Session、Process 和 UsageCollector。

In [ ]:
lab.setup()

## Step 2: Dispatch process

Scheduler 把 READY process 调度为 RUNNING。

In [ ]:
lab.dispatch()

## Step 3: Inspect model-visible request

模型只看到当前请求，不拥有 Process 状态或预算 authority。

In [ ]:
lab.show_model_request()

## Step 4: Ask deterministic or real model

模型响应会产生 usage；usage 被记录到 Process，不写成 durable truth。

In [ ]:
lab.model_step()

## Step 5: Safe point budget check

在 LLM call 后的 safe point，Scheduler 根据预算阻塞 Process。

In [ ]:
lab.safe_point_budget_check()

## Step 6: Recover from budget pause

重置 runtime usage 后 Process 回到 READY；Agent 和 Session 身份不变。

In [ ]:
lab.recover_after_budget_pause()

## Summary

这个实验回答：Process 是 runtime identity，Agent 仍是 capability principal。

In [ ]:
lab.summary()
lab.close()